# AnimationStudio — Phase 6: Story Engine & Narrative Intelligence System on Google Colab

This notebook verifies and regenerates the **Phase-6 story engine** output
(`PHASE6.md` / `PHASE6_REPORT.md`) described by `scripts/generate_phase6.py`.

The Story Engine is the studio's narrative brain: it turns curriculum goals,
themes, characters, worlds, and story grammars into validated episode
blueprints — no monolithic LLM prompt required.

This is a **pure-Python / CPU** phase, exactly like Phase 4:

- **No GPU / ComfyUI / model download required** (runs on the free CPU runtime).
- **No Drive mount needed** — nothing here writes `catalog.db`.
- Output: a refreshed `PHASE6_REPORT.md` (doc↔code consistency, 5 sample
  episodes, a Season-1 plan, the quality checklist) plus a green test run.

## What gets verified

| Deliverable | Where |
| --- | --- |
| Curriculum areas (25) | `src/story_engine/curriculum.py` |
| Learning objectives (72) | `src/story_engine/learning_objective.py` |
| Themes (28) | `src/story_engine/theme.py` |
| Characters & relationships | `src/story_engine/casting.py` |
| World zones / locations / weather | `src/story_engine/world.py` |
| Story grammars (14) | `src/story_engine/grammar_data.py` |
| Conflicts / resolutions | `src/story_engine/plot.py` |
| Dialogue / interaction / song / emotion / humor | `dialogue.py`, `interaction.py`, `song.py` |
| Continuity & diversity tracking | `continuity.py`, `diversity.py` |
| Episode generator + validation engine | `generator.py`, `validation.py` |
| Series / season / holiday planner | `planner.py` |

## Steps

1. In Cell 1 set `REPO_URL` to your GitHub clone URL.
2. Runtime -> Run all.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (push master there first).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "master"  #@param ["master", "colab-gpu"]

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"

# Cell 7: push the refreshed report back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# GitHub PAT (Settings -> Developer settings -> Tokens). Needs Contents:
# Read+Write. Leave empty if the repo is public and you push over HTTPS creds.
GITHUB_TOKEN = ""  #@param {type:"string"}


In [ ]:
#@title 2. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    # Full clone so Cell 7 can push the refreshed report back.
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn"])
print("Studio installed (branch:", BRANCH, ")")
print("No GPU needed for Phase 6 — CPU runtime is fine.")


In [ ]:
#@title 3. Preview the Story Engine library inventory (no generation)

from src.story_engine import (
    CurriculumEngine, ThemeEngine, LearningObjectiveEngine, CharacterEngine,
    WorldEngine, StoryGrammarLibrary, ConflictEngine,
)

curriculum = CurriculumEngine()
themes = ThemeEngine()
objectives = LearningObjectiveEngine()
characters = CharacterEngine()
world = WorldEngine()
grammars = StoryGrammarLibrary()
conflicts = ConflictEngine()

rows = [
    ("Curriculum areas", len(curriculum.list_areas())),
    ("Themes", len(themes._themes)),
    ("Learning objectives", len(objectives._objectives)),
    ("Characters", len(characters.list_all())),
    ("World zones", len(world.get_all_zones())),
    ("World locations", len(world.get_all_locations())),
    ("Story grammars", len(grammars.list_grammars())),
    ("Conflicts", len(conflicts.conflicts)),
]
width = max(len(n) for n, _ in rows)
for name, count in rows:
    print(f"{name:<{width}} : {count}")

print("\nSample curriculum areas:", ", ".join(curriculum.list_areas()[:6]), "...")
print("Sample grammars:", ", ".join(grammars.list_grammars()[:5]), "...")


In [ ]:
#@title 4. Regenerate PHASE6_REPORT.md (doc↔code consistency + sample episodes + season plan)

# Pure Python — verifies the StoryEngine/ markdown guides against the encoded
# libraries (21 facts), generates 5 sample episodes, validates every episode
# through StoryValidationEngine, plans Season 1, and writes PHASE6_REPORT.md.
run([sys.executable, "scripts/generate_phase6.py"])


In [ ]:
#@title 5. Run the Phase-6 test suites

# tests/test_story_engine.py              — engines, grammars, generator,
#                                           validation, doc↔code consistency.
# tests/test_batch_generator.py           — batch/season episode generation.
# tests/test_story_to_production_integration.py — story_engine -> production
#                                           (Phase 1 character/world/asset) wiring.
#
# NOTE: up to 5 failures in test_story_engine.py inside
# TestExpandedCharacterRoster/TestStoryCatalogIntegration are a DOCUMENTED,
# PRE-EXISTING baseline: this repo's catalog.db is corrupt and those tests
# resolve against it (constraint C-CATALOGDB — never rewrite catalog.db here).
# They are unrelated to the story engine itself and appear identically in the
# mainline suite. A clean catalog.db in a real install lets them pass.
run([sys.executable, "-m", "pytest",
     "tests/test_story_engine.py",
     "tests/test_batch_generator.py",
     "tests/test_story_to_production_integration.py", "-q"])


In [ ]:
#@title 6. Review the report

from IPython.display import Markdown, display

with open(f"{REPO}/PHASE6_REPORT.md") as fh:
    report = fh.read()
display(Markdown(report[:6000]))
print("... (full file:", len(report), "chars)")


In [ ]:
#@title 7. Sync the refreshed report (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import _basic_auth_header

    def _run(cmd, **kw):
        print("+ " + " ".join(cmd))
        return subprocess.run(cmd, check=False, cwd=REPO, **kw)

    _run(["git", "config", "user.name", GIT_NAME])
    _run(["git", "config", "user.email", GIT_EMAIL])
    _run(["git", "add", "PHASE6_REPORT.md"])
    dirty = _run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if dirty.stdout.strip():
        _run(["git", "commit", "-m",
              f"Phase 6 report {datetime.now():%Y-%m-%d %H:%M}"])
        if GITHUB_TOKEN:
            _run(["git", "-c",
                  f"http.extraheader=Authorization: {_basic_auth_header(GITHUB_TOKEN)}",
                  "push", "origin", BRANCH])
        else:
            _run(["git", "push", "origin", BRANCH])
    else:
        print("Report unchanged — nothing to push.")
else:
    from google.colab import files
    files.download(f"{REPO}/PHASE6_REPORT.md")
    print("Downloaded PHASE6_REPORT.md.")


## Next steps

- The refreshed `PHASE6_REPORT.md` now lives in your repo (or was downloaded).
- Browse generated episodes in the **Review UI → Story** pane — every library
  previewed above is rendered there, plus episode detail and validation.
- Song *placement* is decided here (Song Engine → lyrical intent); actual audio
  is produced later by the Phase-5 music pipeline.
- Re-run Cells 4–7 any time `StoryEngine/*.md` or `src/story_engine/` change.
